# Latent Vacuum Stationarity (LVS) — Visualisations Interactives

**Objectif :** Modéliser visuellement les concepts clés du cadre LVS :
- Les **points fixes** du paysage potentiel du vide quantique (= particules stables)
- Le **flot du groupe de renormalisation** et la convergence vers un point fixe GUT
- La **cristallisation cosmique** : l'expansion comme front de stabilisation (réaction-diffusion)
- L'**émergence du temps** : mécanisme de Page-Wootters dans un état global statique
- La **dilatation temporelle** comme proximité du substrat atemporel (métrique de Schwarzschild)

Chaque section contient :
1. Le contexte physique et les équations
2. Le code commenté
3. La visualisation interactive

---

### Principe LVS en une phrase

> *La réalité physique observable correspond à l'ensemble des configurations stationnaires — les points fixes — de l'espace de configuration du vide quantique.*

$$\hat{H}|\Psi\rangle = 0 \quad \text{(Wheeler–DeWitt : l'état global est statique)}$$

In [ ]:
# ============================================================
# INSTALLATION DES DEPENDANCES (décommenter si nécessaire)
# ============================================================
# !pip install numpy scipy matplotlib plotly ipywidgets

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib import cm
from scipy.integrate import odeint, solve_ivp
from scipy.ndimage import laplace
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import HTML, display, Markdown

# Style matplotlib sombre cohérent
plt.style.use('dark_background')
plt.rcParams.update({
    'figure.facecolor': '#0b0f1a',
    'axes.facecolor': '#0b0f1a',
    'axes.edgecolor': '#334455',
    'axes.labelcolor': '#9ca3af',
    'text.color': '#e8e6e1',
    'xtick.color': '#6b7280',
    'ytick.color': '#6b7280',
    'grid.color': '#1a1f35',
    'grid.alpha': 0.5,
    'figure.dpi': 120,
    'font.size': 11,
})

# Palette LVS
GOLD = '#d4a853'
BLUE = '#60a5fa'
CYAN = '#22d3ee'
EMERALD = '#34d399'
ROSE = '#fb7185'
VIOLET = '#a78bfa'
MUTED = '#6b7280'

print("Environnement prêt.")

---
## 1. Paysage Potentiel du Vide — Les Points Fixes QCD

### Concept LVS
Sous LVS, chaque hadron (proton, neutron, baryons exotiques...) est un **point fixe** — une vallée stable dans le paysage potentiel du vide quantique. La **profondeur** de la vallée détermine :
- La **masse** de la particule (énergie d'interaction confinée)
- Sa **stabilité** (durée de vie)

| Particule | Type | Masse (MeV/c²) | Durée de vie | Profondeur du point fixe |
|-----------|------|----------------|-------------|-------------------------|
| Proton | Baryon (uud) | 938 | Stable (> 10³⁴ ans) | Très profond |
| Neutron | Baryon (udd) | 940 | ~15 min | Profond |
| Ξcc⁺ | Baryon (dcc) | 3 620 | < 10⁻¹³ s | Modéré |
| Tcc⁺ | Tétraquark (ccūd̄) | 3 875 | ~10⁻²³ s | Peu profond |
| Pc(4450)⁺ | Pentaquark | 4 450 | ~10⁻²³ s | Peu profond |

### Modèle
On modélise le potentiel effectif $V(x, y)$ comme une surface 2D avec des puits gaussiens :

$$V(x,y) = V_{\text{fond}}(x,y) - \sum_i D_i \, \exp\!\left(-\frac{(x-x_i)^2 + (y-y_i)^2}{2\sigma_i^2}\right)$$

où $D_i$ est la profondeur (proportionnelle à $\log(\tau_i)$, la durée de vie) et $\sigma_i$ la largeur du bassin d'attraction.

In [ ]:
# ============================================================
# SCENE 1 : Paysage potentiel du vide quantique (Plotly 3D)
# ============================================================

# --- Données des hadrons (points fixes QCD) ---
# Chaque entrée : (x, y, profondeur, largeur, nom, masse_MeV, couleur)
# La profondeur est calibrée sur log10(durée_de_vie_en_secondes)
# Proton : stable > 10^34 ans ≈ 10^41.5 s → profondeur max
# Neutron : ~900 s → log10(900) ≈ 2.95
# Ξcc⁺ : < 10^-13 s → log10(10^-13) = -13
# Tcc⁺, Pc : ~10^-23 s → -23

hadrons = [
    # (x, y, depth, sigma, name, mass_MeV, color)
    (-1.8, -1.2,  4.0,  0.65, 'Proton (uud)',      938,  GOLD),
    (-1.5,  1.3,  3.5,  0.60, 'Neutron (udd)',      940,  BLUE),
    ( 1.5, -0.8,  2.0,  0.45, 'Ξcc⁺ (dcc)',       3620,  VIOLET),
    ( 2.0,  1.5,  1.0,  0.35, 'Tcc⁺ (ccūd̄)',     3875,  ROSE),
    ( 0.0,  0.3,  0.8,  0.30, 'Pc(4450)⁺',        4450,  CYAN),
]

# --- Construction de la surface potentielle ---
# Grille 2D représentant un espace de configuration réduit à 2 dimensions
# (en réalité, l'espace de configuration QCD est de dimension infinie)
resolution = 200
x_range = np.linspace(-4, 4, resolution)
y_range = np.linspace(-4, 4, resolution)
X, Y = np.meshgrid(x_range, y_range)

# Potentiel de fond : ondulations douces simulant la topologie du vide
# (dans la réalité, ce serait le potentiel effectif de QCD sur réseau)
V_background = 0.15 * (
    np.sin(X * 0.7) * np.cos(Y * 0.5) +
    0.3 * np.sin(X * 1.2 + Y * 0.9)
)

# Ajout des puits gaussiens (un par hadron = un point fixe)
V = V_background.copy()
for (hx, hy, depth, sigma, name, mass, color) in hadrons:
    # Puits gaussien : profondeur D, largeur sigma
    r_squared = (X - hx)**2 + (Y - hy)**2
    V -= depth * np.exp(-r_squared / (2 * sigma**2))

# --- Simulation de gradient descent (bille qui roule vers un point fixe) ---
# On part d'un point aléatoire et on suit -∇V (descente de gradient)
# Cela illustre comment les configurations non-stationnaires "tombent"
# vers les points fixes stables du paysage

def compute_gradient(V, x_range, y_range):
    """Calcul du gradient numérique du potentiel."""
    dx = x_range[1] - x_range[0]
    dy = y_range[1] - y_range[0]
    dVdy, dVdx = np.gradient(V, dy, dx)
    return dVdx, dVdy

dVdx, dVdy = compute_gradient(V, x_range, y_range)

def gradient_descent_path(x0, y0, V, dVdx, dVdy, x_range, y_range, 
                          n_steps=500, lr=0.02, damping=0.95):
    """
    Simule la trajectoire d'une "bille" roulant sur le paysage potentiel.
    
    Physiquement, cela représente une configuration de champs quantiques
    non-stationnaire qui "relaxe" vers le point fixe le plus proche
    dans l'espace de configuration.
    
    Parameters:
        x0, y0 : position initiale dans l'espace de configuration
        V : potentiel sur la grille
        dVdx, dVdy : gradients du potentiel
        n_steps : nombre de pas de simulation
        lr : taux d'apprentissage (vitesse de descente)
        damping : facteur d'amortissement (friction)
    """
    path = [(x0, y0)]
    vx, vy = 0.0, 0.0  # vitesse initiale nulle
    x, y = x0, y0
    
    for _ in range(n_steps):
        # Interpolation bilinéaire du gradient à la position (x, y)
        # (la grille est discrète, mais la bille se déplace en continu)
        ix = np.clip(int((x - x_range[0]) / (x_range[1] - x_range[0])), 0, len(x_range)-2)
        iy = np.clip(int((y - y_range[0]) / (y_range[1] - y_range[0])), 0, len(y_range)-2)
        
        # Force = -∇V (la bille roule vers les vallées)
        gx = -dVdx[iy, ix]
        gy = -dVdy[iy, ix]
        
        # Mise à jour de la vitesse avec amortissement
        vx = damping * vx + lr * gx
        vy = damping * vy + lr * gy
        
        # Mise à jour de la position
        x += vx
        y += vy
        
        # Borner dans le domaine
        x = np.clip(x, x_range[0], x_range[-1])
        y = np.clip(y, y_range[0], y_range[-1])
        
        path.append((x, y))
        
        # Arrêt si la vitesse est quasi-nulle (point fixe atteint)
        if abs(vx) + abs(vy) < 1e-6:
            break
    
    return np.array(path)

# Générer plusieurs trajectoires partant de points différents
np.random.seed(42)
trajectories = []
for _ in range(8):
    # Points de départ aléatoires sur le bord du paysage
    angle = np.random.uniform(0, 2*np.pi)
    r = 3.0 + np.random.uniform(-0.5, 0.5)
    x0 = r * np.cos(angle)
    y0 = r * np.sin(angle)
    path = gradient_descent_path(x0, y0, V, dVdx, dVdy, x_range, y_range)
    trajectories.append(path)

print(f"Surface : {resolution}x{resolution} = {resolution**2} points")
print(f"Trajectoires calculées : {len(trajectories)}")
print(f"Points fixes (hadrons) : {len(hadrons)}")

In [ ]:
# --- Visualisation 3D interactive (matplotlib) ---
# La surface représente le paysage potentiel V(x,y)
# Les vallées profondes sont les points fixes stables (hadrons)

# Colormap personnalisée : sombre → bleu → or
lvs_landscape_cmap = LinearSegmentedColormap.from_list('lvs_land', [
    (0.0, '#1a0a2e'),   # Puits profonds → violet sombre
    (0.3, '#0d47a1'),   # Puits modérés → bleu profond
    (0.5, '#1565c0'),   # Intermédiaire
    (0.7, '#2e7d32'),   # Surface → vert
    (0.85, '#f9a825'),  # Crêtes → or
    (1.0, '#ffcc02'),   # Sommets → jaune vif
])

fig = plt.figure(figsize=(14, 9))
ax = fig.add_subplot(111, projection='3d')

# 1) Surface potentielle
# stride réduit pour performance, rstride/cstride = pas d'échantillonnage
surf = ax.plot_surface(
    X, Y, V,
    cmap=lvs_landscape_cmap,
    alpha=0.85,
    rstride=2, cstride=2,
    edgecolor='none',
    antialiased=True,
)
fig.colorbar(surf, ax=ax, shrink=0.5, pad=0.08, label='V(x,y)')

# 2) Marqueurs au fond de chaque puits (= position des points fixes)
marker_colors = [GOLD, BLUE, VIOLET, ROSE, CYAN]
for i, (hx, hy, depth, sigma, name, mass, color) in enumerate(hadrons):
    # Évaluer le potentiel exact au fond du puits
    v_min = -depth + 0.15 * (np.sin(hx*0.7)*np.cos(hy*0.5) + 0.3*np.sin(hx*1.2+hy*0.9))
    ax.scatter([hx], [hy], [v_min + 0.05], color=color, s=100, 
               marker='D', zorder=5, edgecolors='white', linewidths=0.5)
    ax.text(hx, hy, v_min + 0.25, f'{name}\n{mass} MeV',
            color=color, fontsize=7, ha='center', fontweight='bold')

# 3) Trajectoires de gradient descent (billes roulant vers les points fixes)
traj_colors = ['#22d3ee', '#f472b6', '#a3e635', '#fbbf24', 
               '#818cf8', '#fb923c', '#67e8f9', '#c084fc']
for i, path in enumerate(trajectories):
    z_path = []
    for (px, py) in path:
        ix = np.clip(int((px - x_range[0])/(x_range[1]-x_range[0])), 0, resolution-1)
        iy = np.clip(int((py - y_range[0])/(y_range[1]-y_range[0])), 0, resolution-1)
        z_path.append(V[iy, ix] + 0.03)
    ax.plot(path[:, 0], path[:, 1], z_path, 
            color=traj_colors[i % len(traj_colors)], linewidth=1.5, alpha=0.8)

# Style
ax.set_xlabel('Coord. config. 1', fontsize=10, labelpad=10)
ax.set_ylabel('Coord. config. 2', fontsize=10, labelpad=10)
ax.set_zlabel('V(x,y)', fontsize=10, labelpad=10)
ax.set_title('Paysage Potentiel du Vide — Points Fixes QCD', color=GOLD, fontsize=14, pad=20)
ax.view_init(elev=35, azim=-60)
ax.set_facecolor('#06080f')
fig.patch.set_facecolor('#0b0f1a')

# Légende manuelle
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='D', color='w', markerfacecolor=GOLD, markersize=8, label='Proton (938 MeV) — très profond', linestyle=''),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=BLUE, markersize=8, label='Neutron (940 MeV) — profond', linestyle=''),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=VIOLET, markersize=8, label='Ξcc⁺ (3620 MeV) — modéré', linestyle=''),
    Line2D([0], [0], marker='D', color='w', markerfacecolor=ROSE, markersize=8, label='Tcc⁺ (3875 MeV) — peu profond', linestyle=''),
    Line2D([0], [0], color=CYAN, linewidth=2, label='Trajectoires → points fixes'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=8, framealpha=0.3)

plt.tight_layout()
plt.show()


### Interprétation LVS

- Les **vallées profondes** (proton, neutron) sont des points fixes très stables : ces particules existent "longtemps" (le proton est peut-être éternellement stable).
- Les **cuvettes peu profondes** (Tcc⁺, Pc(4450)⁺) sont des points fixes réels mais fragiles — ils "existent" fugacement ($\sim 10^{-23}$ s).
- Les **trajectoires** montrent des configurations de champs non-stationnaires qui "relaxent" vers le point fixe le plus proche — c'est le mécanisme de sélection LVS.
- La masse n'est pas une propriété intrinsèque mais la **profondeur du point fixe** = énergie d'interaction confinée.

> *"Sous LVS, l'existence n'est pas binaire mais graduée : la profondeur du point fixe détermine le degré de manifestation."*

---

## 2. Flot du Groupe de Renormalisation — Convergence vers le Point Fixe

### Concept LVS
Les constantes de couplage du Modèle Standard **courent** avec l'échelle d'énergie $\mu$. Le flot est gouverné par les fonctions bêta $\beta(g)$, et les **points fixes** ($\beta(g) = 0$) correspondent à des théories invariantes d'échelle.

LVS propose que ce que le RG décrit localement — le flot vers des points fixes à certaines échelles — est un principe ontologique global : **la réalité observable est le point fixe de l'espace de configuration du vide.**

### Équations (1-loop du Modèle Standard)

L'évolution des 3 constantes de couplage de jauge avec l'énergie :

$$\frac{d\alpha_i}{d \ln\mu} = \frac{b_i}{2\pi} \alpha_i^2$$

avec les coefficients 1-loop du SM :
- $b_1 = +\frac{41}{10}$ (U(1) — augmente avec l'énergie)
- $b_2 = -\frac{19}{6}$ (SU(2) — diminue : liberté asymptotique)
- $b_3 = -7$ (SU(3) — diminue : liberté asymptotique, QCD)

La solution analytique 1-loop donne :
$$\alpha_i^{-1}(\mu) = \alpha_i^{-1}(M_Z) - \frac{b_i}{2\pi}\ln\frac{\mu}{M_Z}$$

In [ ]:
# ============================================================
# SCENE 2 : Flot du Groupe de Renormalisation (matplotlib)
# ============================================================

# --- Constantes expérimentales à l'échelle M_Z = 91.2 GeV ---
# (Particle Data Group 2024)
M_Z = 91.2  # masse du Z en GeV

# Constantes de couplage normalisées GUT (convention SU(5))
# alpha_1 = (5/3) * alpha_Y (hypercharge normalisée)
alpha_1_MZ = (5/3) * 0.01017  # ≈ 0.0170
alpha_2_MZ = 0.03378           # couplage faible
alpha_3_MZ = 0.1185            # couplage fort (QCD)

# --- Coefficients bêta 1-loop du Modèle Standard ---
# Ces coefficients déterminent comment chaque force "court" avec l'énergie
b1 = 41/10    # U(1) : le couplage AUGMENTE (pas de liberté asymptotique)
b2 = -19/6    # SU(2) : le couplage DIMINUE (liberté asymptotique)
b3 = -7        # SU(3) : le couplage DIMINUE fortement (QCD confine !)

# --- Solution analytique 1-loop ---
# alpha_i^{-1}(mu) = alpha_i^{-1}(M_Z) - b_i/(2*pi) * ln(mu/M_Z)
# On travaille en échelle log : t = ln(mu/M_Z)

def alpha_inv(alpha_MZ, b, log_mu_over_MZ):
    """
    Calcule alpha^{-1}(mu) à partir de la solution 1-loop.
    
    Paramètres:
        alpha_MZ : constante de couplage à l'échelle M_Z
        b : coefficient bêta 1-loop
        log_mu_over_MZ : ln(mu/M_Z)
    
    Retourne:
        alpha^{-1}(mu)
    """
    return 1/alpha_MZ - b / (2 * np.pi) * log_mu_over_MZ

# Échelle d'énergie : de 10^1 GeV (basse énergie) à 10^17 GeV (au-delà GUT)
log_mu = np.linspace(np.log(10), np.log(1e17), 1000)  # ln(mu) en GeV
log_mu_over_MZ = log_mu - np.log(M_Z)  # ln(mu/M_Z)

# Calcul des 3 couplages inversés en fonction de l'énergie
a1_inv = alpha_inv(alpha_1_MZ, b1, log_mu_over_MZ)
a2_inv = alpha_inv(alpha_2_MZ, b2, log_mu_over_MZ)
a3_inv = alpha_inv(alpha_3_MZ, b3, log_mu_over_MZ)

# Convertir ln(mu) en log10(mu) pour l'axe x (plus lisible)
log10_mu = log_mu / np.log(10)

# --- Trouver le point de quasi-convergence ---
# On cherche l'énergie où les 3 courbes se rapprochent le plus
spread = np.abs(a1_inv - a2_inv) + np.abs(a2_inv - a3_inv) + np.abs(a1_inv - a3_inv)
idx_min = np.argmin(spread)
convergence_energy = log10_mu[idx_min]
convergence_alpha_inv = (a1_inv[idx_min] + a2_inv[idx_min] + a3_inv[idx_min]) / 3

print(f"Quasi-convergence à : 10^{convergence_energy:.1f} GeV")
print(f"α⁻¹ moyen au point de convergence : {convergence_alpha_inv:.1f}")
print(f"Écart résiduel : {spread[idx_min]:.2f} (unification imparfaite dans le SM pur)")

In [ ]:
# --- Visualisation du flot RG ---

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7), gridspec_kw={'width_ratios': [2, 1]})

# === Panneau gauche : α⁻¹(μ) vs log₁₀(μ) ===
# C'est la représentation classique du flot RG

ax1.plot(log10_mu, a1_inv, color=ROSE, linewidth=2.5, label=r'$\alpha_1^{-1}$ — U(1) électromagnétique')
ax1.plot(log10_mu, a2_inv, color=BLUE, linewidth=2.5, label=r'$\alpha_2^{-1}$ — SU(2) faible')
ax1.plot(log10_mu, a3_inv, color=EMERALD, linewidth=2.5, label=r'$\alpha_3^{-1}$ — SU(3) forte (QCD)')

# Zone de quasi-convergence GUT
ax1.axvspan(14, 17, alpha=0.1, color=GOLD, label='Zone GUT (~10¹⁵⁻¹⁷ GeV)')

# Marqueur du point de convergence
ax1.plot(convergence_energy, convergence_alpha_inv, '*', 
         color=GOLD, markersize=20, zorder=5, label=f'Quasi-convergence (10^{convergence_energy:.1f} GeV)')

# Échelles physiques remarquables
scales = [
    (np.log10(M_Z), r'$M_Z$', MUTED),
    (np.log10(173), r'$m_t$ (top)', MUTED),
    (3, r'LHC ($10^3$ GeV)', CYAN),
]
for (lmu, label, c) in scales:
    ax1.axvline(lmu, color=c, linestyle=':', alpha=0.4, linewidth=1)
    ax1.text(lmu + 0.1, 62, label, color=c, fontsize=8, rotation=90, va='top')

ax1.set_xlabel(r'$\log_{10}(\mu / \mathrm{GeV})$ — Échelle d\'énergie', fontsize=12)
ax1.set_ylabel(r'$\alpha_i^{-1}(\mu)$ — Couplage inversé', fontsize=12)
ax1.set_title('Flot du Groupe de Renormalisation (1-loop SM)', color=GOLD, fontsize=14)
ax1.legend(loc='upper left', fontsize=9, framealpha=0.3)
ax1.set_xlim(1, 17)
ax1.set_ylim(0, 70)
ax1.grid(True, alpha=0.2)

# Annotation LVS
ax1.annotate(
    'Point fixe LVS ?\n(β = 0 pour les 3 forces)',
    xy=(convergence_energy, convergence_alpha_inv),
    xytext=(convergence_energy - 3, convergence_alpha_inv + 15),
    fontsize=10, color=GOLD,
    arrowprops=dict(arrowstyle='->', color=GOLD, lw=1.5),
    bbox=dict(boxstyle='round,pad=0.3', facecolor='#1a1f35', edgecolor=GOLD, alpha=0.8),
)

# === Panneau droit : Diagramme de phase α₂ vs α₃ ===
# Visualise le "flot" dans l'espace des couplages
# Le point fixe est là où les trajectoires convergent

# Convertir α⁻¹ en α pour le diagramme de phase
a2_vals = np.where(a2_inv > 0, 1/a2_inv, np.nan)
a3_vals = np.where(a3_inv > 0, 1/a3_inv, np.nan)

# Colorer par l'énergie (basse → haute)
points = ax2.scatter(a3_vals, a2_vals, c=log10_mu, cmap='plasma', s=2, alpha=0.8)
plt.colorbar(points, ax=ax2, label=r'$\log_{10}(\mu/\text{GeV})$', shrink=0.8)

# Direction du flot (flèches)
n_arrows = 10
step = len(log10_mu) // n_arrows
for i in range(0, len(log10_mu) - step, step):
    if np.isnan(a3_vals[i]) or np.isnan(a2_vals[i]): continue
    if np.isnan(a3_vals[i+step]) or np.isnan(a2_vals[i+step]): continue
    dx = a3_vals[i+step] - a3_vals[i]
    dy = a2_vals[i+step] - a2_vals[i]
    ax2.annotate('', xy=(a3_vals[i+step], a2_vals[i+step]),
                xytext=(a3_vals[i], a2_vals[i]),
                arrowprops=dict(arrowstyle='->', color=GOLD, lw=1, alpha=0.6))

ax2.set_xlabel(r'$\alpha_3$ (couplage fort)', fontsize=11)
ax2.set_ylabel(r'$\alpha_2$ (couplage faible)', fontsize=11)
ax2.set_title('Espace des phases (α₃, α₂)', color=GOLD, fontsize=12)
ax2.grid(True, alpha=0.2)

# Marqueur du point de convergence
ax2.plot(a3_vals[idx_min], a2_vals[idx_min], '*', color=GOLD, markersize=15, zorder=5)
ax2.annotate('Point fixe', xy=(a3_vals[idx_min], a2_vals[idx_min]),
            xytext=(a3_vals[idx_min]+0.005, a2_vals[idx_min]+0.005),
            fontsize=10, color=GOLD)

plt.tight_layout()
plt.show()

print("\n→ Le flot RG montre les 3 couplages convergeant vers une même valeur")
print("  à haute énergie. Sous LVS, ce point de convergence est le POINT FIXE")
print("  dont les coordonnées déterminent TOUTES les constantes physiques.")

### Interprétation LVS

- Le **panneau gauche** montre les 3 constantes de couplage qui "courent" avec l'énergie et quasi-convergent vers $\sim 10^{15-16}$ GeV.
- Le **panneau droit** est un diagramme de phase : on voit le flot dans l'espace (α₃, α₂) converger vers un point.
- Sous LVS : les constantes ne sont pas des paramètres libres "choisis" — ce sont les **coordonnées du point fixe** dans l'espace de configuration du vide.
- L'unification imparfaite dans le SM pur suggère que notre cartographie du point fixe est incomplète (nouvelles physiques à découvrir ?).

---

## 3. Front de Cristallisation Cosmique — Réaction-Diffusion

### Concept LVS
Sous LVS, l'expansion cosmologique n'est pas l'étirement d'un espace pré-existant. C'est l'**actualisation progressive** du vide non-manifesté en points fixes stables. L'inflation serait une cascade rapide de stabilisation.

### Modèle : Équation de Fisher-KPP
On utilise l'équation de **Fisher-KPP** (réaction-diffusion), un modèle classique de propagation de fronts stables :

$$\frac{\partial u}{\partial t} = D \nabla^2 u + r \, u(1 - u)$$

où :
- $u(x,y,t) \in [0, 1]$ : degré d'actualisation (0 = vide non-manifesté, 1 = point fixe stable)
- $D$ : coefficient de diffusion (vitesse de propagation du front)
- $r$ : taux de réaction (vitesse de stabilisation locale)

La solution développe un **front de propagation** à vitesse $c_{\text{front}} = 2\sqrt{Dr}$ qui sépare le vide non-actualisé ($u \approx 0$) du vide cristallisé ($u \approx 1$).

**Analogie LVS :** Le front est le bord de l'univers observable — là où le vide "cristallise" en structures stables.

In [ ]:
# ============================================================
# SCENE 3 : Cristallisation cosmique (Fisher-KPP 2D)
# ============================================================

# --- Paramètres de la simulation ---
N = 300            # taille de la grille NxN
D = 1.0            # coefficient de diffusion
r = 0.5            # taux de réaction (vitesse de stabilisation)
dt = 0.1           # pas de temps
dx = 1.0           # pas spatial
n_steps = 400      # nombre total de pas de temps
n_snapshots = 8    # nombre d'instantanés à capturer

# --- Condition initiale ---
# Petit noyau actualisé au centre (= Big Bang / germe initial)
u = np.zeros((N, N))
cx, cy = N // 2, N // 2
radius_init = 5  # rayon du germe initial en cellules

# Germe gaussien au centre
Y_grid, X_grid = np.meshgrid(np.arange(N), np.arange(N))
r_grid = np.sqrt((X_grid - cx)**2 + (Y_grid - cy)**2)
u = np.exp(-r_grid**2 / (2 * radius_init**2))

# --- Perturbation : ajout de "germes" aléatoires ---
# Simulant des fluctuations quantiques du vide qui nucléent
# des points fixes spontanément en avance du front principal
np.random.seed(12)
n_seeds = 15
for _ in range(n_seeds):
    sx = np.random.randint(50, N-50)
    sy = np.random.randint(50, N-50)
    if r_grid[sx, sy] > 30:  # seulement loin du centre
        u[sx-2:sx+3, sy-2:sy+3] = np.random.uniform(0.3, 0.7)

# --- Résolution numérique par différences finies ---
# Schéma d'Euler explicite : u(t+dt) = u(t) + dt * [D * Lap(u) + r * u * (1-u)]

snapshots = []  # stockage des instantanés
snap_interval = n_steps // n_snapshots

print("Simulation Fisher-KPP 2D en cours...")
for step in range(n_steps):
    # Laplacien discret (différences finies 5-points)
    lap = laplace(u) / (dx**2)
    
    # Terme de réaction Fisher-KPP : r * u * (1 - u)
    # u=0 : rien ne se passe (vide non-manifesté)
    # u=1 : saturé (point fixe stabilisé)
    # 0 < u < 1 : croissance logistique → stabilisation
    reaction = r * u * (1 - u)
    
    # Mise à jour temporelle (Euler explicite)
    u = u + dt * (D * lap + reaction)
    
    # Borner entre 0 et 1
    u = np.clip(u, 0, 1)
    
    # Sauvegarder des instantanés
    if step % snap_interval == 0:
        snapshots.append((step, u.copy()))

# Ajouter l'état final
snapshots.append((n_steps, u.copy()))

print(f"Simulation terminée : {n_steps} pas, {len(snapshots)} instantanés capturés")
print(f"Vitesse théorique du front : c = 2√(Dr) = {2*np.sqrt(D*r):.2f} cellules/pas")

In [ ]:
# --- Visualisation des instantanés de cristallisation ---

n_show = min(8, len(snapshots))
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

# Colormap personnalisée : noir → bleu → or → blanc
# Noir = vide non-manifesté, Bleu = front d'actualisation,
# Or = point fixe stabilisé, Blanc = structures denses
from matplotlib.colors import LinearSegmentedColormap
lvs_cmap = LinearSegmentedColormap.from_list('lvs', [
    (0.0, '#06080f'),    # Vide non-manifesté (potentialité pure)
    (0.15, '#0d1b3e'),   # Début d'actualisation
    (0.35, '#1565c0'),   # Front d'actualisation (bleu)
    (0.55, '#d4a853'),   # Points fixes stabilisés (or)
    (0.80, '#f5deb3'),   # Structures matures
    (1.0,  '#ffffff'),   # Structures denses (galaxies)
])

for i in range(n_show):
    ax = axes[i]
    step_num, snapshot = snapshots[i]
    
    ax.imshow(snapshot, cmap=lvs_cmap, vmin=0, vmax=1, origin='lower')
    
    # Contour du front (u = 0.5) : c'est le bord de l'univers observable
    ax.contour(snapshot, levels=[0.5], colors=[CYAN], linewidths=0.8, alpha=0.7)
    
    # Titre avec le temps cosmique
    pct_actualized = 100 * np.mean(snapshot > 0.5)
    ax.set_title(f't = {step_num}\n{pct_actualized:.0f}% actualisé', 
                 color=GOLD, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('#06080f')

# Cacher les axes supplémentaires si besoin
for i in range(n_show, len(axes)):
    axes[i].set_visible(False)

fig.suptitle('Cristallisation Cosmique — Front de Stabilisation (Fisher-KPP)',
             color=GOLD, fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

print("\n→ Le front bleu/cyan est la frontière entre le vide non-manifesté (noir)")
print("  et les points fixes actualisés (or). C'est le 'bord' de l'univers.")
print("  Les germes isolés = fluctuations quantiques qui nucléent en avance du front.")

In [ ]:
# --- Profil radial du front de cristallisation ---
# Montre la transition nette entre vide non-manifesté et points fixes stables

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Profil radial moyen pour chaque instantané
r_max = N // 2
r_vals = np.arange(r_max)

for i, (step_num, snapshot) in enumerate(snapshots):
    # Moyenne angulaire du champ u en fonction de la distance au centre
    profile = np.zeros(r_max)
    counts = np.zeros(r_max)
    for x in range(N):
        for y in range(N):
            d = int(np.sqrt((x - cx)**2 + (y - cy)**2))
            if d < r_max:
                profile[d] += snapshot[x, y]
                counts[d] += 1
    profile = np.where(counts > 0, profile / counts, 0)
    
    alpha = 0.3 + 0.7 * (i / len(snapshots))
    ax1.plot(r_vals, profile, color=GOLD, alpha=alpha, linewidth=1.5,
             label=f't={step_num}' if i % 2 == 0 else '')

ax1.set_xlabel('Distance au centre (cellules)', fontsize=11)
ax1.set_ylabel('u(r) — Degré d\'actualisation', fontsize=11)
ax1.set_title('Profil radial du front de cristallisation', color=GOLD, fontsize=12)
ax1.axhline(0.5, color=CYAN, linestyle='--', alpha=0.5, label='Front (u=0.5)')
ax1.legend(fontsize=9, framealpha=0.3)
ax1.grid(True, alpha=0.2)
ax1.set_ylim(-0.05, 1.1)

# Rayon du front vs temps (position de u=0.5)
front_radii = []
for step_num, snapshot in snapshots:
    profile = np.zeros(r_max)
    counts = np.zeros(r_max)
    for x in range(N):
        for y in range(N):
            d = int(np.sqrt((x-cx)**2 + (y-cy)**2))
            if d < r_max:
                profile[d] += snapshot[x,y]
                counts[d] += 1
    profile = np.where(counts > 0, profile/counts, 0)
    # Trouver r où profile ≈ 0.5
    idx_front = np.argmin(np.abs(profile - 0.5))
    front_radii.append((step_num, r_vals[idx_front]))

steps_arr = [s for s,_ in front_radii]
radii_arr = [r for _,r in front_radii]

ax2.plot(steps_arr, radii_arr, 'o-', color=CYAN, linewidth=2, markersize=6)

# Ajustement linéaire (la vitesse du front Fisher-KPP est constante)
if len(steps_arr) > 2:
    coeffs = np.polyfit(steps_arr, radii_arr, 1)
    fit_line = np.polyval(coeffs, steps_arr)
    ax2.plot(steps_arr, fit_line, '--', color=GOLD, alpha=0.7,
             label=f'Vitesse du front : {coeffs[0]:.2f} cell/pas')
    ax2.legend(fontsize=10, framealpha=0.3)

ax2.set_xlabel('Temps de simulation', fontsize=11)
ax2.set_ylabel('Rayon du front (cellules)', fontsize=11)
ax2.set_title('Expansion : rayon du front vs temps', color=GOLD, fontsize=12)
ax2.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\n→ Le front se propage à vitesse quasi-constante : c ≈ {coeffs[0]:.2f} cellules/pas")
print(f"   (théorie Fisher-KPP : c = 2√(Dr) = {2*np.sqrt(D*r):.2f})")
print("→ Sous LVS, c'est l'analogue de l'expansion cosmologique :")
print("   l'actualisation progressive du vide en points fixes stables.")

### Interprétation LVS

- Le **noir** est le vide non-manifesté — potentialité pure, pas d'espace, pas de temps.
- Le **front bleu** est la frontière d'actualisation — là où le vide "cristallise" en points fixes.
- Les **zones or** sont les points fixes stabilisés — matière, structures, espace-temps manifesté.
- Les **germes isolés** sont des fluctuations quantiques qui nucléent en avance du front (analogues aux galaxies précoces vues par JWST).
- L'**inflation** correspondrait à une phase où le taux de réaction $r$ est très grand → cascade ultra-rapide de stabilisation.

---

## 4. Émergence du Temps — Mécanisme de Page-Wootters

### Concept LVS
L'équation de Wheeler-DeWitt dit : $\hat{H}|\Psi\rangle = 0$. L'univers **n'évolue pas**. L'état global est statique.

Le mécanisme de **Page-Wootters** (1983) montre comment le temps peut **émerger** :
- On divise l'univers en 2 sous-systèmes : une **horloge** (C) et un **système** (S)
- L'état global est un état intriqué statique : $|\Psi\rangle = \sum_t |t\rangle_C \otimes |\phi(t)\rangle_S$
- Quand on conditionne sur l'horloge $|t\rangle_C$, le système S "évolue" : $|\phi(t)\rangle_S$ change avec $t$
- Le temps est un **paramètre relationnel**, pas une dimension fondamentale

### Modèle
On simule un système bipartite avec :
- **Horloge** : qubit avec Hamiltonien $H_C = \omega \sigma_z$
- **Système** : qubit avec Hamiltonien $H_S = \epsilon \sigma_x$
- Contrainte : $H_{\text{total}} = H_C \otimes I + I \otimes H_S = 0$ sur l'état global

In [ ]:
# ============================================================
# SCENE 4 : Emergence du temps (Page-Wootters)
# ============================================================

# --- Le mecanisme de Page-Wootters ---
# L'univers est decrit par un etat STATIQUE |Psi> satisfaisant H|Psi> = 0.
# On divise l'univers en : Horloge (C) a N niveaux + Systeme (S) qubit.
# Le temps emerge quand on conditionne |Psi> sur l'etat de l'horloge.

N_clock = 32   # nombre de niveaux de l'horloge (resolution temporelle)
N_sys = 2      # systeme = qubit

# --- Hamiltonien de l'horloge ---
# Oscillateur a N niveaux : H_C[n,n] = n * omega
omega = 2 * np.pi / N_clock

H_clock = np.diag(np.arange(N_clock) * omega)

# --- Hamiltonien du systeme ---
# Qubit avec precession : H_S = epsilon * sigma_x
sigma_x = np.array([[0, 1], [1, 0]], dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]], dtype=complex)
sigma_z = np.array([[1, 0], [0, -1]], dtype=complex)
I_sys = np.eye(N_sys, dtype=complex)
I_clock = np.eye(N_clock, dtype=complex)

epsilon = omega
H_sys = epsilon * sigma_x

# --- Hamiltonien total (contrainte) ---
H_total = np.kron(H_clock, I_sys) + np.kron(I_clock, H_sys)

print(f"Espace de Hilbert total : {N_clock} x {N_sys} = {N_clock * N_sys} dimensions")

# --- Construction de l'etat Page-Wootters ("history state") ---
# |Psi> = (1/sqrt(N)) sum_n |n>_C (x) e^{-i H_S n/omega} |phi_0>_S
# Cet etat encode TOUTE l'evolution du systeme dans un etat statique.

phi_0 = np.array([1, 0], dtype=complex)  # etat initial : spin-up

Psi_global = np.zeros(N_clock * N_sys, dtype=complex)
for n in range(N_clock):
    # Operateur d'evolution : e^{-i epsilon sigma_x n/omega}
    # Pour sigma_x : e^{-i theta sigma_x} = cos(theta)I - i sin(theta)sigma_x
    theta = epsilon * n / omega  # = n (car epsilon = omega)
    U_n = np.cos(theta) * I_sys - 1j * np.sin(theta) * sigma_x
    phi_n = U_n @ phi_0
    Psi_global[n * N_sys : (n+1) * N_sys] = phi_n

Psi_global /= np.linalg.norm(Psi_global)

residual = np.linalg.norm(H_total @ Psi_global)
print(f"||H|Psi>|| = {residual:.4f} (residu de la contrainte)")
print(f"L'etat global encode TOUTE l'histoire dans une structure atemporelle.")


In [ ]:
# --- Emergence du temps via conditionnement sur l'horloge ---

# === Methode (a) : conditionnement discret sur |n> ===
expect_x_disc = []
expect_y_disc = []
expect_z_disc = []

for n in range(N_clock):
    # Extraire |phi(n)>_S = <n|_C (x) I_S |Psi>
    phi_n = Psi_global[n * N_sys : (n+1) * N_sys]
    norm = np.linalg.norm(phi_n)
    if norm > 1e-10:
        phi_n = phi_n / norm
    rho_n = np.outer(phi_n, phi_n.conj())
    expect_x_disc.append(np.real(np.trace(rho_n @ sigma_x)))
    expect_y_disc.append(np.real(np.trace(rho_n @ sigma_y)))
    expect_z_disc.append(np.real(np.trace(rho_n @ sigma_z)))

expect_x_disc = np.array(expect_x_disc)
expect_y_disc = np.array(expect_y_disc)
expect_z_disc = np.array(expect_z_disc)

# === Methode (b) : conditionnement continu ===
# |theta>_C = (1/sqrt(N)) sum_n e^{in*theta} |n>
n_times = 200
theta_vals = np.linspace(0, 4 * np.pi, n_times)

expect_x = []
expect_y = []
expect_z = []

for theta in theta_vals:
    clock_phases = np.exp(1j * np.arange(N_clock) * theta) / np.sqrt(N_clock)
    phi_S = np.zeros(N_sys, dtype=complex)
    for n in range(N_clock):
        phi_S += clock_phases[n].conj() * Psi_global[n * N_sys : (n+1) * N_sys]
    norm = np.linalg.norm(phi_S)
    if norm > 1e-10:
        phi_S = phi_S / norm
    rho_S = np.outer(phi_S, phi_S.conj())
    expect_x.append(np.real(np.trace(rho_S @ sigma_x)))
    expect_y.append(np.real(np.trace(rho_S @ sigma_y)))
    expect_z.append(np.real(np.trace(rho_S @ sigma_z)))

expect_x = np.array(expect_x)
expect_y = np.array(expect_y)
expect_z = np.array(expect_z)

print("Conditionnement sur l'horloge calcule.")
print(f"<sigma_x> discret oscille entre {expect_x_disc.min():.3f} et {expect_x_disc.max():.3f}")
print(f"<sigma_z> discret oscille entre {expect_z_disc.min():.3f} et {expect_z_disc.max():.3f}")
print(f"Le systeme 'evolue' quand on scanne l'horloge, MAIS |Psi> ne change JAMAIS.")


In [ ]:
# --- Visualisation de l'emergence du temps ---

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# === (a) Observables vs temps discret ===
ax = axes[0, 0]
time_disc = np.arange(N_clock)
ax.plot(time_disc, expect_x_disc, 'o-', color=ROSE, linewidth=1.5, markersize=4,
        label=r'$\langle\sigma_x\rangle$')
ax.plot(time_disc, expect_y_disc, 'o-', color=BLUE, linewidth=1.5, markersize=4,
        label=r'$\langle\sigma_y\rangle$')
ax.plot(time_disc, expect_z_disc, 'o-', color=EMERALD, linewidth=1.5, markersize=4,
        label=r'$\langle\sigma_z\rangle$')
ax.set_xlabel("Niveau d'horloge n (temps discret)", fontsize=11)
ax.set_ylabel("Valeur d'attente", fontsize=11)
ax.set_title('(a) Le systeme evolue quand on scanne l horloge', color=GOLD, fontsize=12)
ax.legend(fontsize=10, framealpha=0.3)
ax.grid(True, alpha=0.2)
ax.axhline(0, color=MUTED, linestyle='-', alpha=0.3)

# === (b) Sphere de Bloch ===
ax = axes[0, 1]
scatter = ax.scatter(expect_x, expect_z, c=theta_vals, cmap='plasma', s=5, alpha=0.8)
plt.colorbar(scatter, ax=ax, label=r'$\theta$ (temps continu)')
circle = plt.Circle((0, 0), 1, fill=False, color=MUTED, linestyle='--', alpha=0.3)
ax.add_patch(circle)
ax.scatter(expect_x_disc, expect_z_disc, c=time_disc, cmap='plasma',
           s=50, edgecolors='white', linewidths=0.5, zorder=5, marker='D')
ax.set_xlabel(r'$\langle\sigma_x\rangle$', fontsize=11)
ax.set_ylabel(r'$\langle\sigma_z\rangle$', fontsize=11)
ax.set_title('(b) Trajectoire sur la sphere de Bloch', color=GOLD, fontsize=12)
ax.set_aspect('equal')
ax.set_xlim(-1.3, 1.3)
ax.set_ylim(-1.3, 1.3)
ax.grid(True, alpha=0.2)

# === (c) Matrice densite globale (constante !) ===
ax = axes[1, 0]
rho_global = np.outer(Psi_global, Psi_global.conj())
block_size = min(16, N_clock * N_sys)
im = ax.imshow(np.abs(rho_global[:block_size, :block_size]), cmap='inferno', vmin=0)
plt.colorbar(im, ax=ax, label=r'$|\rho_{ij}|$')
ax.set_title('(c) Matrice densite GLOBALE (statique !)', color=GOLD, fontsize=12)
ax.set_xlabel('Composante')
ax.set_ylabel('Composante')
ax.text(block_size/2, -2, r'$\hat{H}|\Psi\rangle = 0$ : cet etat ne change JAMAIS',
        ha='center', fontsize=10, color=ROSE,
        bbox=dict(boxstyle='round', facecolor='#1a1f35', edgecolor=ROSE, alpha=0.8))

# === (d) Analogie du livre ===
ax = axes[1, 1]
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title("(d) Analogie : le Livre de l'Univers", color=GOLD, fontsize=12)

n_pages = 12
current_page = 5
for i in range(n_pages):
    x = 0.5 + i * 0.7
    is_current = (i == current_page)
    fc = GOLD if is_current else '#1a1f35'
    ec = GOLD if is_current else '#334455'
    alp = 0.8 if is_current else 0.3
    rect = plt.Rectangle((x, 1), 0.5, 3.5, facecolor=fc, edgecolor=ec,
                          alpha=alp, linewidth=1.5 if is_current else 0.5)
    ax.add_patch(rect)
    ax.text(x + 0.25, 0.6, f't={i}', ha='center', fontsize=7, color=MUTED)

obs_x_pos = 0.5 + current_page * 0.7 + 0.25
ax.annotate('VOUS\n(observateur)', xy=(obs_x_pos, 1), xytext=(obs_x_pos, 0.1),
           fontsize=9, color=CYAN, ha='center',
           arrowprops=dict(arrowstyle='->', color=CYAN, lw=2))
ax.text(5, 5.2, 'Toutes les pages existent simultanement',
        ha='center', fontsize=11, color='#e8e6e1')
ax.text(5, 4.8, 'Le temps = votre position de lecture',
        ha='center', fontsize=10, color=GOLD)

plt.tight_layout()
plt.show()

print()
print("RESULTAT CLE : l'etat global |Psi> est STATIQUE (H|Psi> = 0).")
print("Mais quand on conditionne sur l'horloge, le systeme SEMBLE evoluer.")
print("Le temps est un artefact de l'observation locale dans un tout atemporel.")
print("L'univers est sur PAUSE : il l'a toujours ete.")


### Interprétation LVS

- **(a)** Le système semble évoluer (les valeurs d'attente oscillent) quand on scanne l'horloge.
- **(b)** La trajectoire sur la sphère de Bloch montre cette "évolution" — mais c'est une corrélation statique, pas un mouvement.
- **(c)** La matrice densité globale est **figée**. Rien ne change. $\hat{H}|\Psi\rangle = 0$.
- **(d)** L'analogie du livre : toutes les pages existent. Le temps est votre position de lecture.

> *"Le temps est le coût relationnel d'être un observateur local dans un tout atemporel."*

---

## 5. Dilatation Temporelle — Profondeur du Point Fixe

### Concept LVS
Sous LVS, la masse est l'énergie confinée d'un point fixe. Plus le point fixe est profond, plus on est proche du substrat atemporel. La dilatation gravitationnelle du temps est littéralement la **transition vers l'atemporalité**.

### Métrique de Schwarzschild
La dilatation du temps dans un champ gravitationnel est donnée par la métrique de Schwarzschild :

$$d\tau = dt \sqrt{1 - \frac{r_s}{r}}$$

où :
- $d\tau$ : temps propre (horloge locale)
- $dt$ : temps coordonnée (horloge à l'infini)
- $r_s = 2GM/c^2$ : rayon de Schwarzschild
- $r$ : distance au centre

Quand $r \to r_s$ (horizon) : $d\tau/dt \to 0$ → **le temps s'arrête** → retour au substrat atemporel.

In [ ]:
# ============================================================
# SCENE 5 : Dilatation temporelle (Schwarzschild)
# ============================================================

# --- Constantes physiques ---
G = 6.674e-11       # m³/(kg·s²)
c = 2.998e8          # m/s
M_sun = 1.989e30     # kg

# --- Objets astrophysiques réels ---
objects = [
    # (nom, masse_en_M_sun, rayon_surface_en_m, couleur)
    ('Terre',           3.003e-6,   6.371e6,   BLUE),
    ('Naine blanche',   0.6,        8.0e6,     '#e8e6e1'),
    ('Étoile à neutrons', 1.4,      1.1e4,     VIOLET),
    ('Trou noir 10 M☉', 10,         None,      ROSE),  # rayon = r_s
    ('Sgr A* (4M M☉)',  4e6,        None,      GOLD),  # trou noir supermassif
]

# Calcul du rayon de Schwarzschild et du facteur de dilatation
print("Objets astrophysiques et dilatation temporelle :\n")
print(f"{'Objet':<25} {'Masse (M☉)':<15} {'r_s (m)':<15} {'r_surface (m)':<15} {'dτ/dt surface':<15} {'Profondeur LVS'}")
print("-" * 105)

obj_data = []
for name, M_ratio, R_surface, color in objects:
    M = M_ratio * M_sun
    r_s = 2 * G * M / c**2  # rayon de Schwarzschild
    
    if R_surface is None:
        R_surface = r_s  # pour les trous noirs, la surface = horizon
    
    # Facteur de dilatation à la surface : sqrt(1 - r_s/r)
    if R_surface > r_s:
        dilation_factor = np.sqrt(1 - r_s / R_surface)
    else:
        dilation_factor = 0.0  # à l'horizon ou en-dessous
    
    # "Profondeur LVS" = 1 - dilation_factor (0 = espace plat, 1 = horizon)
    depth_lvs = 1 - dilation_factor
    
    print(f"{name:<25} {M_ratio:<15.2e} {r_s:<15.3e} {R_surface:<15.3e} {dilation_factor:<15.6f} {depth_lvs:.6f}")
    obj_data.append((name, r_s, R_surface, dilation_factor, depth_lvs, color))

In [ ]:
# --- Visualisation principale : le puits gravitationnel ---

fig, axes = plt.subplots(1, 3, figsize=(18, 7))

# === (a) Facteur de dilatation vs distance (plusieurs objets) ===
ax = axes[0]

for name, r_s, R_surf, dil, depth, color in obj_data:
    # Profil radial de la dilatation : dτ/dt = sqrt(1 - r_s/r)
    r_min = r_s * 1.001  # juste au-dessus de l'horizon
    r_max_plot = r_s * 50
    r_vals = np.linspace(r_min, r_max_plot, 500)
    dilation = np.sqrt(1 - r_s / r_vals)
    
    # Normaliser l'axe x en unités de r_s pour comparaison
    ax.plot(r_vals / r_s, dilation, color=color, linewidth=2, label=name)
    
    # Marqueur à la surface
    if R_surf > r_s:
        ax.plot(R_surf / r_s, np.sqrt(1 - r_s / R_surf), 'o', 
                color=color, markersize=8, zorder=5)

# Horizon (r = r_s)
ax.axvline(1.0, color=ROSE, linestyle='--', alpha=0.5, label='Horizon (r = r_s)')
ax.axhline(0, color=ROSE, linestyle=':', alpha=0.3)

# Zone LVS
ax.fill_between([1, 2], 0, 1, alpha=0.05, color=ROSE)
ax.text(1.3, 0.1, 'Zone\natemporelle', color=ROSE, fontsize=9, ha='center')

ax.set_xlabel(r'$r / r_s$ (distance en rayons de Schwarzschild)', fontsize=11)
ax.set_ylabel(r'$d\tau/dt = \sqrt{1 - r_s/r}$', fontsize=11)
ax.set_title('(a) Dilatation temporelle', color=GOLD, fontsize=12)
ax.set_xlim(1, 20)
ax.set_ylim(-0.05, 1.05)
ax.legend(fontsize=8, framealpha=0.3, loc='lower right')
ax.grid(True, alpha=0.2)

# === (b) "Profondeur LVS" : analogie du puits potentiel ===
ax = axes[1]

# On dessine un puits de potentiel gravitationnel
# avec les objets placés à leur profondeur respective
r_norm = np.linspace(1.01, 50, 500)
# Potentiel effectif (Schwarzschild) : Φ = -r_s/(2r) en unités normalisées
phi = -1 / (2 * r_norm)  # potentiel newtonien normalisé

ax.plot(r_norm, phi, color=MUTED, linewidth=1, alpha=0.5)
ax.fill_between(r_norm, phi, -0.6, alpha=0.05, color=BLUE)

# Placer chaque objet
for name, r_s, R_surf, dil, depth, color in obj_data:
    # Position x = R_surface/r_s, y = -profondeur
    x_pos = max(1.5, R_surf / r_s) if r_s > 0 else 50
    if x_pos > 50: x_pos = 40  # Terre est très loin en r/r_s
    y_pos = -depth * 0.5
    ax.plot(x_pos, y_pos, 'o', color=color, markersize=12, zorder=5)
    ax.text(x_pos, y_pos - 0.04, name, color=color, fontsize=8, 
            ha='center', va='top')

# Horizon
ax.axhline(-0.5, color=ROSE, linestyle='--', alpha=0.4)
ax.text(25, -0.52, 'Horizon → Substrat atemporel', color=ROSE, fontsize=9, ha='center')

ax.set_xlabel(r'$r/r_s$', fontsize=11)
ax.set_ylabel('Profondeur du point fixe', fontsize=11)
ax.set_title('(b) Puits gravitationnel = profondeur LVS', color=GOLD, fontsize=12)
ax.set_xlim(0, 50)
ax.set_ylim(-0.6, 0.05)
ax.grid(True, alpha=0.2)

# === (c) Horloges à différentes profondeurs ===
ax = axes[2]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('(c) Horloges à différentes profondeurs', color=GOLD, fontsize=12)

# Dessiner des horloges avec des vitesses différentes
clock_data = [
    ('Espace plat', 0.0, 1.0, EMERALD),
    ('Surface Terre', 0.01, 0.9999999993, BLUE),
    ('Naine blanche', 0.1, 0.95, '#e8e6e1'),
    ('Étoile neutrons', 0.7, 0.55, VIOLET),
    ('Près horizon', 0.99, 0.1, ROSE),
]

# Temps de simulation pour les aiguilles
import time as time_module
t_now = 2.5  # angle fixe pour la figure statique

for i, (label, depth_frac, speed, color) in enumerate(clock_data):
    cx = 2 + i * 1.5
    cy = 5
    radius = 0.6
    
    # Cadran
    circle = plt.Circle((cx, cy), radius, fill=False, color=color, linewidth=1.5, alpha=0.7)
    ax.add_patch(circle)
    
    # Aiguille (angle proportionnel à la vitesse)
    angle = t_now * speed * 2  # l'aiguille tourne proportionnellement à speed
    ax.plot([cx, cx + radius * 0.7 * np.sin(angle)],
            [cy, cy + radius * 0.7 * np.cos(angle)],
            color=color, linewidth=2)
    
    # Label
    ax.text(cx, cy - radius - 0.3, label, ha='center', fontsize=7, color=color)
    ax.text(cx, cy - radius - 0.6, f'×{speed:.2f}', ha='center', fontsize=8, 
            color=GOLD, fontweight='bold')

# Flèche de profondeur
ax.annotate('', xy=(8.5, 4), xytext=(1, 4),
           arrowprops=dict(arrowstyle='->', color=GOLD, lw=2))
ax.text(5, 3.5, 'Profondeur croissante → Temps ralentit → Atemporalité', 
        ha='center', fontsize=10, color=GOLD)

ax.text(5, 8, 'LVS : la dilatation temporelle est la\ntransition vers le substrat atemporel',
        ha='center', fontsize=11, color='#e8e6e1',
        bbox=dict(boxstyle='round', facecolor='#1a1f35', edgecolor=GOLD, alpha=0.8))

ax.text(5, 1.5, 'Au trou noir : le temps CESSE d\'émerger\n→ retour à l\'espace de configuration pur',
        ha='center', fontsize=10, color=ROSE, style='italic')

plt.tight_layout()
plt.show()

print("\n→ La dilatation gravitationnelle n'est pas un 'ralentissement' mystérieux.")
print("  C'est l'approche du substrat atemporel — la couche fondamentale")
print("  où le temps n'a jamais émergé.")
print("  Le trou noir est un point fixe si profond que le temps s'y éteint.")

### Interprétation LVS

- **(a)** Le facteur de dilatation $d\tau/dt$ tombe vers zéro à l'horizon. C'est la même courbe pour tous les objets, seule l'échelle de $r_s$ change.
- **(b)** Le puits gravitationnel **est** le puits du point fixe. Plus l'objet est massif, plus le puits est profond, plus on est proche de l'atemporalité.
- **(c)** Les horloges tournent de plus en plus lentement en s'enfonçant. À l'horizon, l'horloge s'arrête — le temps cesse d'émerger.

> *"Un trou noir est un point fixe si profond qu'à l'horizon, la dilatation temporelle devient infinie. L'intérieur représente un retour à l'espace de configuration non-manifesté."*

---

## Synthèse

| Visualisation | Physique réelle | Lecture LVS |
|---|---|---|
| **Paysage 3D** | Potentiel QCD avec masses hadroniques | Les particules sont des vallées stables du vide |
| **Flot RG** | β-fonctions 1-loop du SM | Les constantes sont des coordonnées d'un point fixe |
| **Cristallisation** | Fisher-KPP (réaction-diffusion) | L'expansion est l'actualisation progressive du vide |
| **Page-Wootters** | État bipartite H|Ψ⟩=0 | Le temps émerge des corrélations dans un état statique |
| **Schwarzschild** | Métrique GR réelle | La dilatation = approche du substrat atemporel |

### L'univers est sur pause.
Il l'a toujours été. Ce que nous appelons "temps" est le coût d'être un observateur local dans une structure atemporelle.

---
*Fabien Music Polly — Mars 2026*